In [0]:
%run ../common/config

In [0]:
claims = spark.table(f"{env_catalog}.silver.claims")
transactions = spark.table(f"{env_catalog}.silver.claims_transactions")
encounters = spark.table(f"{env_catalog}.silver.encounters")

In [0]:
from pyspark.sql.functions import *

In [0]:
from pyspark.sql.functions import current_timestamp

fact_claims = (
    claims.alias("c")
    .join(
        transactions.alias("t"),
        "claim_id",
        "left"
    )
    .join(
        encounters.alias("e"),
        claims.patient_id == encounters.patient_id,
        "left"
    )
    .select(
        col("c.claim_id").alias("claim_id"),
        col("c.patient_id").alias("patient_id"),
        col("c.provider_id").alias("provider_id"),
        col("c.service_date").alias("service_date"),

        col("t.transaction_id").alias("transaction_id"),
        col("t.transaction_type").alias("transaction_type"),

        col("t.amount").alias("amount"),
        col("t.payments").alias("payments"),
        col("t.adjustments").alias("adjustments"),
        col("t.transfers").alias("transfers"),
        col("t.outstanding").alias("outstanding"),

        col("t.procedure_code").alias("procedure_code"),

        col("e.encounter_id").alias("encounter_id"),
        col("e.encounter_class").alias("encounter_class"),

        col("e.reason_description").alias("reason_description")
    )
    .withColumn(
        "gold_load_timestamp",
        current_timestamp()
    )
)

In [0]:
fact_claims.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{env_catalog}.gold.fact_claims")